# CBD Adapter — MediaPipe Overlay Video

**`adapter/` role of the [Common Behavior Data](https://github.com/Koichi3333/common-behavior-data) project.**
Target: **an annotated video on the original pixels**.

This notebook reads the canonical dataset written by
`cbd_generator_video_to_cbd.ipynb` and draws it back onto the source frames:
pose, hands, face, tracked objects, gestures and interaction candidates.
It never runs MediaPipe — every number on screen comes out of
`04_behavior_dataset/`.

```text
cbd_dataset.zip ─▶ [ THIS NOTEBOOK ] ─▶ output/01_mediapipe_overlay/mediapipe_overlay.mp4
 (canonical CBD)     draw only                                     + adapter_report.json
```

## Adapter rules honoured here

1. **Reads canonical data**, never another adapter's output
2. **Converts at its boundary** — canonical is Y-up/right-handed, but an
   overlay lives in image space, so this adapter deliberately reads the
   *image-space* series (`human/pose_landmarks.csv`, `human/hand_landmarks.csv`,
   `human/face_landmarks.jsonl`, `objects/object_tracks.csv`), which is the
   raw observation the pixels were measured in
3. Model and motion separation does not apply — the "model" here is the video
4. **Does not promote candidates** — `grasp_candidate` is drawn as
   "Grasp Candidate", never as "Grasp"
5. **Reports what it could not represent** in `adapter_report.json`
6. **Preserves provenance** — interpolated (undetected) object boxes are drawn
   dashed and labelled `interp`

## Input

Any one of these, tried in order — no editing needed:

1. the dataset already unpacked in this runtime (the generator just ran here)
2. `/content/cbd_dataset.zip` (`colab upload`)
3. the Colab upload widget (UI only)

## How to run — colab CLI

```bash
# same session as the generator: nothing to upload
colab exec -s cbd -f cbd_adapter_mediapipe_overlay.ipynb --timeout 1800
colab download -s cbd \
  /content/human_behavior_demo_2_0/output/01_mediapipe_overlay/mediapipe_overlay.mp4 \
  ./mediapipe_overlay.mp4

# fresh session
colab upload -s cbd2 ./cbd_dataset.zip /content/cbd_dataset.zip
colab exec   -s cbd2 -f cbd_adapter_mediapipe_overlay.ipynb --timeout 1800
```

In the Colab UI just run the cells top to bottom; the finished video is
embedded below `[A3]`. In a headless run the player is skipped and the path is
printed instead, so `colab exec` always terminates.


In [ ]:
# @title [A1] Environment setup
# =====================================================================
# Overlay adapter 1/4.
# This adapter only draws, so it needs no MediaPipe, no MuJoCo and no GPU:
# OpenCV and ffmpeg (both preinstalled in Colab) are enough.
# =====================================================================
import json
import shutil
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# Run mode: Colab UI or headless (colab CLI / Run all / papermill)
# ---------------------------------------------------------------
# The two entry points differ in exactly one way that matters here: whether
# the kernel accepts stdin. `colab exec` calls execute_code() without
# allow_stdin, so an upload widget would hang until the run times out, and an
# embedded base64 video player would dump megabytes into the log. Detect the
# mode once, then degrade gracefully instead of blocking.
def _stdin_available():
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or not hasattr(shell, "kernel"):
            return False
        return bool(getattr(shell.kernel, "_allow_stdin", False))
    except Exception:  # noqa: BLE001
        return False


IS_COLAB_UI = _stdin_available()
SHOW_MEDIA = IS_COLAB_UI      # embed videos / images only in the browser UI
RUN_MODE = "colab-ui" if IS_COLAB_UI else "headless (colab CLI / Run all)"
print(f"Run mode: {RUN_MODE}")
print("opencv:", cv2.__version__)
_ffmpeg = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print("ffmpeg:", (_ffmpeg.stdout.splitlines() or ["not found"])[0])

# ------------------------- Shared media utilities -------------------------
def encode_h264(source_path, destination_path, fps):
    """Transcode to H.264 with ffmpeg so the video plays in the browser."""
    command = ["ffmpeg", "-y", "-loglevel", "error", "-r", str(fps),
               "-i", str(source_path), "-c:v", "libx264",
               "-pix_fmt", "yuv420p", "-movflags", "+faststart",
               str(destination_path)]
    result = subprocess.run(command, capture_output=True)
    if result.returncode != 0 or not Path(destination_path).exists():
        shutil.copy(source_path, destination_path)
    return str(destination_path)


def show_video(video_path, width=520):
    """Embed a video player -- Colab UI only.

    In a headless run (colab CLI) the base64 payload would be written to the
    execution log, so print the path instead.
    """
    if not SHOW_MEDIA:
        size_mb = Path(video_path).stat().st_size / 1024 / 1024
        print(f"[preview skipped in headless mode] {video_path} ({size_mb:.1f} MB)")
        return
    import base64
    from IPython.display import HTML, display
    data = Path(video_path).read_bytes()
    if len(data) > 40 * 1024 * 1024:
        print(f"Video too large to embed, skipping preview: {video_path} "
              f"({len(data) / 1024 / 1024:.1f} MB)")
        return
    encoded = base64.b64encode(data).decode()
    display(HTML(f'<video width="{width}" controls loop playsinline '
                 f'src="data:video/mp4;base64,{encoded}"></video>'))


def make_video_writer(path, fps, size):
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"),
                             fps, size)
    if not writer.isOpened():
        raise RuntimeError(f"Could not create the output video: {path}")
    return writer


def even(value):
    return int(value) - int(value) % 2


In [ ]:
# @title [A2] Load the Common Behavior Data
# ---------------------------------------------------------------
# Locate the Common Behavior Data
# ---------------------------------------------------------------
# Resolution order, so the same cell works from either entry point:
#   1. an already-unpacked dataset in this runtime (the generator just ran)
#   2. cbd_dataset.zip sitting on the VM (`colab upload`)
#   3. the upload widget -- Colab UI only
#   4. otherwise: stop with the exact command needed, instead of hanging
CBD_DIR_OVERRIDE = ""      # optional: a directory containing 04_behavior_dataset/

PROJECT_DIR = Path("/content/human_behavior_demo_2_0")
OUTPUT_DIR = PROJECT_DIR / "output"
WORK_DIR = PROJECT_DIR / "_work"
ZIP_CANDIDATES = ["/content/cbd_dataset.zip", "cbd_dataset.zip",
                  str(PROJECT_DIR / "cbd_dataset.zip")]


def _find_dataset(root):
    """Return the 04_behavior_dataset directory below root, if any."""
    root = Path(root)
    direct = root / "output/04_behavior_dataset/manifest.json"
    if direct.exists():
        return direct.parent
    if (root / "manifest.json").exists() and root.name == "04_behavior_dataset":
        return root
    hit = next(iter(sorted(root.glob("**/04_behavior_dataset/manifest.json"))), None)
    return hit.parent if hit else None


def _unpack(zip_path):
    print(f"Unpacking {zip_path} -> {PROJECT_DIR}")
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(PROJECT_DIR)


DATASET_DIR = _find_dataset(CBD_DIR_OVERRIDE) if CBD_DIR_OVERRIDE else None
if DATASET_DIR is None:
    DATASET_DIR = _find_dataset(PROJECT_DIR)
if DATASET_DIR is None:
    _zip = next((p for p in ZIP_CANDIDATES if Path(p).exists()), None)
    if _zip is None and IS_COLAB_UI:
        print("No Common Behavior Data on this VM. Upload cbd_dataset.zip "
              "(produced by cbd_generator_video_to_cbd.ipynb).")
        from google.colab import files
        _uploaded = files.upload()
        _zip = "/content/" + next(iter(_uploaded))
    if _zip is None:
        raise RuntimeError(
            "No Common Behavior Data on this VM and this is a headless run, "
            "so no upload widget can be opened.\n"
            "Run the generator on this session first, or upload its output:\n"
            "    colab upload -s <session> ./cbd_dataset.zip "
            "/content/cbd_dataset.zip")
    _unpack(_zip)
    DATASET_DIR = _find_dataset(PROJECT_DIR)
if DATASET_DIR is None:
    raise RuntimeError("cbd_dataset.zip does not contain 04_behavior_dataset/")

OUTPUT_DIR = DATASET_DIR.parent
# Normal layout: <project>/output/04_behavior_dataset. A flat bundle (the
# dataset directly under the extract root) is accepted too -- then the extract
# root doubles as the project root, so nothing is written outside it.
PROJECT_DIR = OUTPUT_DIR.parent if OUTPUT_DIR.name == "output" else OUTPUT_DIR
WORK_DIR = PROJECT_DIR / "_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_VIDEO = PROJECT_DIR / "source/source_video.mp4"
if not SOURCE_VIDEO.exists():
    SOURCE_VIDEO = next(iter(sorted(PROJECT_DIR.glob("**/source_video.mp4"))),
                        SOURCE_VIDEO)

print("Common Behavior Data:", DATASET_DIR)

# ---------------------------------------------------------------
# Read the canonical timeline
# ---------------------------------------------------------------
# Everything below comes from files. No MediaPipe, no re-computation: that is
# the whole point of the adapter boundary.
MANIFEST = json.loads((DATASET_DIR / "manifest.json").read_text(encoding="utf-8"))
SUMMARY = json.loads(
    (DATASET_DIR / "behavior_summary.json").read_text(encoding="utf-8"))
GENERATOR_CONFIG = {}
if (PROJECT_DIR / "config.json").exists():
    GENERATOR_CONFIG = json.loads(
        (PROJECT_DIR / "config.json").read_text(encoding="utf-8"))

FRAMES = [json.loads(line) for line
          in (DATASET_DIR / "timeline/frames.jsonl").read_text(
              encoding="utf-8").splitlines() if line.strip()]
NUM_FRAMES = len(FRAMES)
if NUM_FRAMES == 0:
    raise RuntimeError("timeline/frames.jsonl is empty")
FPS = float(MANIFEST["fps"])
DT = 1.0 / FPS
TIMESTAMPS = np.array([f["timestamp_sec"] for f in FRAMES], dtype=np.float32)
SOURCE_FRAME_INDEX = [f["source_frame_index"] for f in FRAMES]

BONE_ORDER = (MANIFEST.get("bone_order")
              or list(FRAMES[0]["human"]["bone_rotations_xyzw"]))
FINGER_ORDER = MANIFEST.get("finger_order",
                            ["thumb", "index", "middle", "ring", "little"])

# Bone rotations: stored xyzw in the file, used wxyz by the quaternion helpers
BONE_ROT = {}
for bone in BONE_ORDER:
    xyzw = np.array([f["human"]["bone_rotations_xyzw"][bone] for f in FRAMES],
                    dtype=np.float64)
    BONE_ROT[bone] = np.column_stack([xyzw[:, 3], xyzw[:, 0], xyzw[:, 1],
                                      xyzw[:, 2]])
HIPS_POS = np.array([f["human"]["hips_position"] for f in FRAMES],
                    dtype=np.float64)

# Finger curls [rad]. The timeline stores null on frames where the hand was
# not detected; fill from the nearest valid frame so a renderer does not snap
# the hand open, and record how many frames that affected.
ADAPTER_NOTES = []       # what this adapter could not represent, or had to fill
CURLS, HAND_PRESENT, CURLS_FILLED = {}, {}, {}
for _side in ["left", "right"]:
    raw = [f["human"]["finger_curls_rad"].get(_side) for f in FRAMES]
    present = np.array([r is not None for r in raw])
    values = np.zeros((NUM_FRAMES, len(FINGER_ORDER)))
    if present.any():
        valid_idx = np.where(present)[0]
        for fi in range(NUM_FRAMES):
            src = fi if present[fi] else valid_idx[np.argmin(np.abs(valid_idx - fi))]
            values[fi] = raw[src]
    CURLS[_side.capitalize()] = values
    HAND_PRESENT[_side.capitalize()] = present
    CURLS_FILLED[_side.capitalize()] = int((~present).sum()) if present.any() else 0
    if present.any() and CURLS_FILLED[_side.capitalize()]:
        ADAPTER_NOTES.append(
            f"finger curls: {CURLS_FILLED[_side.capitalize()]}/{NUM_FRAMES} "
            f"{_side} frames had no hand detection and were filled from the "
            "nearest detected frame")
    if not present.any():
        ADAPTER_NOTES.append(f"finger curls: no {_side} hand was ever detected")

PHASE = [f["phase"] for f in FRAMES]
INTERACTIONS = [f["interactions"] for f in FRAMES]
CAPTIONS = [f.get("caption") for f in FRAMES]

# Primary object. "primary_object" is defined on every frame; datasets written
# before that field existed are read back from the objects[] list instead.
PRIMARY_LABEL = (MANIFEST.get("primary", {}).get("object_label")
                 or (SUMMARY.get("primary_object") or {}).get("label"))
PRIMARY_TRACK_ID = (MANIFEST.get("primary", {}).get("object_track_id")
                    or (SUMMARY.get("primary_object") or {}).get("track_id"))
PRIMARY_HAND = (MANIFEST.get("primary", {}).get("hand")
                or SUMMARY.get("primary_hand") or "right")

OBJ_PROXY, OBJ_SOURCE = None, ["none"] * NUM_FRAMES
if PRIMARY_TRACK_ID:
    positions, sources, missing = [], [], 0
    for f in FRAMES:
        block = f.get("primary_object")
        if block is None:
            block = next((o for o in f["objects"]
                          if o.get("track_id") == PRIMARY_TRACK_ID
                          and "proxy_canonical" in o), None)
        if block and block.get("proxy_canonical"):
            positions.append(block["proxy_canonical"])
            sources.append(block.get("position_source", "unknown"))
        else:
            positions.append(positions[-1] if positions else [0.0, 0.9, 0.3])
            sources.append("carried_forward_by_adapter")
            missing += 1
    OBJ_PROXY = np.array(positions, dtype=np.float64)
    OBJ_SOURCE = sources
    if missing:
        ADAPTER_NOTES.append(
            f"object proxy: {missing}/{NUM_FRAMES} frames had no canonical "
            "position in the dataset and were carried forward by this adapter")

print(f"frames={NUM_FRAMES}  fps={FPS:.2f}  duration={TIMESTAMPS[-1]:.2f}s")
print(f"task={SUMMARY.get('task')}  primary_hand={PRIMARY_HAND}  "
      f"primary_object={PRIMARY_LABEL or 'none'}")
print("events:", SUMMARY.get("events") or "none")
print("captions:", sum(1 for c in CAPTIONS if c), "frames carry a caption")
for note in ADAPTER_NOTES:
    print("  note:", note)


def write_adapter_report(path, adapter, target, reads, unrepresented):
    """Rule 5: say what could not be represented, rather than approximating
    it silently. Rule 6: say where every derived value came from."""
    report = {
        "adapter": adapter,
        "target": target,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "source_dataset": {
            "manifest": str(Path(DATASET_DIR) / "manifest.json"),
            "dataset_version": MANIFEST.get("dataset_version"),
            "frame_count": NUM_FRAMES,
            "fps": FPS,
        },
        "reads": reads,
        "filled_or_interpolated": list(ADAPTER_NOTES),
        "not_represented": unrepresented,
        "promoted_candidates": False,
    }
    Path(path).write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("Wrote:", path)
    return report
# ---------------------------------------------------------------
# Image-space view, for drawing
# ---------------------------------------------------------------
# The canonical timeline is Y-up metric space; the overlay needs the raw
# image-space observation the pixels were measured in. Both live in the same
# dataset, side by side -- reading the image-space series here is the
# coordinate conversion at this adapter's boundary.
pose_df = pd.read_csv(DATASET_DIR / "human/pose_landmarks.csv")
POSE_LANDMARK_COUNT = int(pose_df["landmark_id"].max()) + 1
POSE_IMG = np.full((NUM_FRAMES, POSE_LANDMARK_COUNT, 2), np.nan)
POSE_IMG[pose_df["frame"].to_numpy(), pose_df["landmark_id"].to_numpy(), 0] = \
    pose_df["x"].to_numpy()
POSE_IMG[pose_df["frame"].to_numpy(), pose_df["landmark_id"].to_numpy(), 1] = \
    pose_df["y"].to_numpy()
POSE_SEEN = ~np.isnan(POSE_IMG[:, 0, 0])

hand_df = pd.read_csv(DATASET_DIR / "human/hand_landmarks.csv")
HAND_IMG, HAND_SEEN = {}, {}
for side in ["Left", "Right"]:
    array = np.full((NUM_FRAMES, 21, 2), np.nan)
    sub = hand_df[hand_df["hand"] == side.lower()]
    if len(sub):
        array[sub["frame"].to_numpy(), sub["landmark_id"].to_numpy(), 0] = \
            sub["x"].to_numpy()
        array[sub["frame"].to_numpy(), sub["landmark_id"].to_numpy(), 1] = \
            sub["y"].to_numpy()
    HAND_IMG[side] = array
    HAND_SEEN[side] = ~np.isnan(array[:, 0, 0])

FACE_IMG = {}
face_path = DATASET_DIR / "human/face_landmarks.jsonl"
if face_path.exists():
    for line in face_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        FACE_IMG[record["frame"]] = np.asarray(record["landmarks"],
                                               dtype=np.float32)

# Objects: object_tracks.csv is the column-oriented view of the same tracks
# that frames.jsonl carries, and it holds the tracked bbox the overlay needs.
tracks_df = pd.read_csv(DATASET_DIR / "objects/object_tracks.csv")
HAS_TRACK_BBOX = {"bbox_x", "bbox_y", "bbox_width",
                  "bbox_height"}.issubset(tracks_df.columns)
OBJECTS_BY_FRAME = [[] for _ in range(NUM_FRAMES)]
for row in tracks_df.to_dict("records"):
    fi = int(row["frame"])
    if not 0 <= fi < NUM_FRAMES:
        continue
    entry = {
        "track_id": row["track_id"],
        "label": row["label"],
        "role": row.get("role", "target"),
        "center": (float(row["center_x"]), float(row["center_y"])),
        "state": row.get("motion_state", "stationary"),
        "detected": bool(row.get("detected", True)),
        "score": (None if pd.isna(row.get("score", np.nan))
                  else float(row["score"])),
        "is_primary": bool(row.get("is_primary",
                                   row["track_id"] == PRIMARY_TRACK_ID)),
        "bbox": ((float(row["bbox_x"]), float(row["bbox_y"]),
                  float(row["bbox_width"]), float(row["bbox_height"]))
                 if HAS_TRACK_BBOX else None),
    }
    OBJECTS_BY_FRAME[fi].append(entry)

if not HAS_TRACK_BBOX:
    ADAPTER_NOTES.append(
        "object boxes: this dataset predates the bbox columns in "
        "object_tracks.csv, so objects are drawn as centre markers only")

print(f"pose frames: {int(POSE_SEEN.sum())}/{NUM_FRAMES}   "
      f"left hand: {int(HAND_SEEN['Left'].sum())}   "
      f"right hand: {int(HAND_SEEN['Right'].sum())}   "
      f"face: {len(FACE_IMG)}   "
      f"object rows: {len(tracks_df)}")


In [ ]:
# @title [A3] Render the overlay video
# =====================================================================
# Overlay adapter 3/4.
# The behavior data drawn back onto the original pixels: pose / hands / face /
# objects / gestures / interaction candidates.
# =====================================================================
from collections import deque

# ---- Adapter-side settings (they describe the drawing, not the data) ----
OUTPUT_WIDTH = 640          # width of the overlay video
TRAIL_SECONDS = 2.0         # how long the primary object's trail stays

# Skeleton edge lists: a rendering concern, so they live with the renderer
POSE_BONES = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (24, 26), (26, 28), (27, 31), (28, 32),
]
HAND_BONES = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20), (0, 17),
]

HAND_COLOR = {"Left": (255, 160, 60), "Right": (60, 160, 255)}   # BGR
POSE_COLOR = (90, 220, 90)
FACE_COLOR = (200, 200, 120)
OBJ_COLOR = (60, 60, 230)
TRAIL_COLOR = (80, 200, 255)

OVERLAY_DIR = OUTPUT_DIR / "01_mediapipe_overlay"
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

capture = cv2.VideoCapture(str(SOURCE_VIDEO))
if not capture.isOpened():
    raise RuntimeError(
        f"Could not open the source video: {SOURCE_VIDEO}\n"
        "It travels inside cbd_dataset.zip -- re-upload the zip if this "
        "runtime only has the dataset directory.")
SOURCE_WIDTH = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
SOURCE_HEIGHT = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

W = even(min(OUTPUT_WIDTH, SOURCE_WIDTH))
H = even(SOURCE_HEIGHT * W / SOURCE_WIDTH)

raw_overlay = WORK_DIR / "_overlay_raw.mp4"
writer = make_video_writer(raw_overlay, FPS, (W, H))
trail = deque(maxlen=max(2, int(FPS * TRAIL_SECONDS)))
frame_lookup = {source_index: fi
                for fi, source_index in enumerate(SOURCE_FRAME_INDEX)}
MAX_SOURCE_FRAME = max(SOURCE_FRAME_INDEX)


def put_label(image, text, x, y, color=(255, 255, 255), scale=0.45):
    cv2.putText(image, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale,
                (0, 0, 0), 3, cv2.LINE_AA)
    cv2.putText(image, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale,
                color, 1, cv2.LINE_AA)


def draw_box(image, x0, y0, x1, y1, color, dashed=False):
    """Solid box = the detector saw the object on this frame.
    Dashed box = the position was interpolated across a short gap. The
    distinction is in the data (`detected`), so it stays visible here."""
    if not dashed:
        cv2.rectangle(image, (x0, y0), (x1, y1), color, 2)
        return
    step = 10
    for x in range(x0, x1, step * 2):
        cv2.line(image, (x, y0), (min(x + step, x1), y0), color, 2)
        cv2.line(image, (x, y1), (min(x + step, x1), y1), color, 2)
    for y in range(y0, y1, step * 2):
        cv2.line(image, (x0, y), (x0, min(y + step, y1)), color, 2)
        cv2.line(image, (x1, y), (x1, min(y + step, y1)), color, 2)


source_frame_index = 0
while True:
    success, frame_bgr = capture.read()
    if not success or source_frame_index > MAX_SOURCE_FRAME:
        break
    fi = frame_lookup.get(source_frame_index)
    source_frame_index += 1
    if fi is None:
        continue
    record = FRAMES[fi]
    image = cv2.resize(frame_bgr, (W, H))

    # ---- Face mesh (subsampled points, the full mesh is too dense to read) ----
    face = FACE_IMG.get(fi)
    if face is not None:
        for point in face[::6]:
            cv2.circle(image, (int(point[0] * W), int(point[1] * H)),
                       1, FACE_COLOR, -1)

    # ---- Pose skeleton ----
    if POSE_SEEN[fi]:
        pn = POSE_IMG[fi]
        for a, b in POSE_BONES:
            cv2.line(image, (int(pn[a][0] * W), int(pn[a][1] * H)),
                     (int(pn[b][0] * W), int(pn[b][1] * H)), POSE_COLOR, 2)
        for point in pn:
            cv2.circle(image, (int(point[0] * W), int(point[1] * H)), 3,
                       POSE_COLOR, -1)

    # ---- Hands + gesture ----
    for side in ["Left", "Right"]:
        if not HAND_SEEN[side][fi]:
            continue
        hn = HAND_IMG[side][fi]
        color = HAND_COLOR[side]
        for a, b in HAND_BONES:
            cv2.line(image, (int(hn[a][0] * W), int(hn[a][1] * H)),
                     (int(hn[b][0] * W), int(hn[b][1] * H)), color, 2)
        for point in hn:
            cv2.circle(image, (int(point[0] * W), int(point[1] * H)), 2,
                       color, -1)
        wx, wy = int(hn[0][0] * W), int(hn[0][1] * H)
        gesture = record["human"]["gestures"].get(side.lower())
        text = f"{side} Hand"
        if gesture and gesture.get("gesture") not in ("None", "", None):
            text += f" | Gesture: {gesture['gesture']}"
        put_label(image, text, min(wx, W - 220), max(16, wy - 8), color)

    # ---- Objects (bbox + track id + trail + motion state) ----
    for entry in OBJECTS_BY_FRAME[fi]:
        center = (int(entry["center"][0] * W), int(entry["center"][1] * H))
        if entry["bbox"] is not None:
            bx, by, bw, bh = entry["bbox"]
            draw_box(image, int(bx * W), int(by * H),
                     int((bx + bw) * W), int((by + bh) * H),
                     OBJ_COLOR, dashed=not entry["detected"])
            label_x, label_y = int(bx * W), max(14, int(by * H) - 22)
        else:
            label_x, label_y = center[0] - 40, max(14, center[1] - 22)
        label_x = max(0, min(label_x, W - 240))
        state = "Moving" if entry["state"] == "moving" else "Stationary"
        put_label(image,
                  f"Object: {entry['label']}  Track: {entry['track_id']}"
                  + ("" if entry["detected"] else "  (interp)"),
                  label_x, label_y, OBJ_COLOR)
        confidence = ("n/a" if entry["score"] is None
                      else f"{entry['score']:.2f}")
        put_label(image, f"Conf: {confidence}  State: {state}  "
                         f"Role: {entry['role']}",
                  label_x, label_y + 16, OBJ_COLOR)
        cv2.circle(image, center, 4, OBJ_COLOR, -1)
        if entry["is_primary"]:
            trail.append(center)
    for t0, t1 in zip(list(trail), list(trail)[1:]):
        cv2.line(image, t0, t1, TRAIL_COLOR, 2)

    # ---- Behavior caption, top-left ----
    # Candidates stay labelled as candidates -- an overlay must not promote
    # "grasp_candidate" into "grasp".
    lines = ["Person 1"]
    if record["interactions"]:
        top = record["interactions"][0]
        lines.append("Interaction: "
                     + top["type"].replace("_", " ").title())
    caption = CAPTIONS[fi]
    if caption and caption.get("en"):
        lines.append("AI caption: " + caption["en"][:44])
    lines.append(f"Frame: {fi}   Time: {record['timestamp_sec']:05.2f} s")
    band_width = 250 if len(lines) < 4 else 380
    cv2.rectangle(image, (0, 0), (band_width, 18 * len(lines) + 10),
                  (0, 0, 0), -1)
    for li, line in enumerate(lines):
        put_label(image, line, 8, 18 + 18 * li)

    writer.write(image)

capture.release()
writer.release()

OVERLAY_MP4 = OVERLAY_DIR / "mediapipe_overlay.mp4"
encode_h264(raw_overlay, OVERLAY_MP4, FPS)
print("Wrote:", OVERLAY_MP4)
show_video(OVERLAY_MP4)


In [ ]:
# @title [A4] Adapter report + acceptance check + package
# =====================================================================
# Overlay adapter 4/4.
# =====================================================================
def print_checks(title, checks):
    print(f"=== Acceptance criteria ({title}) ===")
    for label, ok in checks:
        print(("  [x] " if ok else "  [ ] ") + label)


def package_adapter_output(zip_name, entries):
    """Zip this adapter's output with paths relative to the project root.

    `entries` may hold directories or single files. Unzipping the result into
    /content/human_behavior_demo_2_0/ on another runtime puts every file back
    where the other notebooks expect it.
    """
    zip_path = PROJECT_DIR / zip_name
    members = []
    for entry in entries:
        entry = Path(entry)
        if entry.is_dir():
            members += [p for p in sorted(entry.rglob("*")) if p.is_file()]
        elif entry.is_file():
            members.append(entry)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in members:
            archive.write(path, path.relative_to(PROJECT_DIR))
    print(f"\nAdapter output: {zip_path} "
          f"({zip_path.stat().st_size / 1024 / 1024:.1f} MB, "
          f"{len(members)} files)")
    print(f"  colab download -s <session> {zip_path} ./{zip_name}")
    if SHOW_MEDIA:
        try:
            from google.colab import files as colab_files
            if zip_path.stat().st_size < 200 * 1024 * 1024:
                colab_files.download(str(zip_path))
            else:
                print("  (over 200 MB: use the file pane on the left)")
        except Exception:  # noqa: BLE001
            print("  (automatic download unavailable; use the file pane)")
    return zip_path
REPORT = write_adapter_report(
    OVERLAY_DIR / "adapter_report.json",
    adapter="mediapipe_overlay",
    target="annotated mp4 in source image space",
    reads=["human/pose_landmarks.csv", "human/hand_landmarks.csv",
           "human/face_landmarks.jsonl", "human/gestures.csv (via frames.jsonl)",
           "objects/object_tracks.csv", "timeline/frames.jsonl",
           "source/source_video.mp4"],
    unrepresented=[
        "canonical bone rotations: an overlay draws observed image-space "
        "landmarks, so the derived rotations are not shown",
        "face blendshapes: recorded in the dataset, not drawn",
        "joint angles and motion metrics: not drawn",
        "object depth (proxy_canonical): 2D overlay, so the z estimate is "
        "not visualised",
    ])

CHECKS = [
    ("Input: canonical dataset located",
     (DATASET_DIR / "manifest.json").exists()),
    ("Input: source video available for drawing", SOURCE_VIDEO.exists()),
    ("Output: mediapipe_overlay.mp4 written", OVERLAY_MP4.exists()),
    ("Output: video is non-empty",
     OVERLAY_MP4.exists() and OVERLAY_MP4.stat().st_size > 1024),
    ("Output: adapter_report.json written",
     (OVERLAY_DIR / "adapter_report.json").exists()),
    ("Data: pose drawn on at least one frame", bool(POSE_SEEN.any())),
    ("Data: objects available to draw", len(tracks_df) > 0),
    ("Data: interaction candidates present",
     any(frame["interactions"] for frame in FRAMES)),
]
print_checks("mediapipe overlay adapter", CHECKS)

package_adapter_output("overlay_adapter_output.zip", [OVERLAY_DIR])
print("\nNext: cbd_adapter_mujoco.ipynb and cbd_adapter_unity_vrm.ipynb read "
      "the same dataset. A three-screen comparison of all three, built with a "
      "Unity recording, ships in examples/human-capture/sample_output/"
      "05_comparison/.")
